## 1. Environment Setup

Set up the Google Colab environment, install the required dependencies, and mount Google Drive for storing model artifacts.

In [1]:
!git clone https://github.com/rimfa/shopee-product-matching.git
%cd /content/shopee-product-matching

Cloning into 'shopee-product-matching'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 30 (delta 3), reused 24 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 1.66 MiB | 8.48 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/shopee-product-matching


In [2]:
%pip install -q kagglehub timm albumentations

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 2. Imports & Configuration

Import the required libraries and configure the computing device for EfficientNet-B0 feature extraction.

import os
import numpy as np
import pandas as pd
import cv2
import torch
import timm
import albumentations as A

from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from src.embeddings import get_embeddings

In [9]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
import timm
import albumentations as A

from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from src.embeddings import get_embeddings

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 3. Data Loading

Download the Shopee Product Matching dataset from Kaggle and load the product metadata and image directory.

In [7]:
import kagglehub

kagglehub.login()

root = kagglehub.competition_download("shopee-product-matching")
print(root)

Kaggle credentials successfully validated.


100%|██████████| 1.68G/1.68G [01:21<00:00, 22.3MB/s]

Extracting files...


/root/.cache/kagglehub/competitions/shopee-product-matching
Kaggle credentials set.
Kaggle credentials successfully validated.


In [10]:
train_path = os.path.join(root, 'train.csv')
full_df = pd.read_csv(train_path)

train_images_path = os.path.join(root, 'train_images')
train_images = os.listdir(train_images_path)

In [11]:
print(f"Total products: {len(full_df)}")
print(f"Total images: {len(train_images)}")

full_df.head(2)

Total products: 34250
Total images: 32412


,posting_id,image,image_phash,title,label_group
0,train_129225211,0000a68812bc7e98c42888dfb1c07da0.jpg,94974f937d4c2433,Paper Bag Victoria Secret,249114794
1,train_3386243561,00039780dfc94d01db8676fe789ecd05.jpg,af3f9460c2838f0f,"Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...",2937985045


## 4. Train / Validation Split

Split the dataset by `label_group` to prevent the same product groups from appearing in both training and validation sets.

- Group-based split
- Validation size: 20% of unique groups
- Random state: 101

In [12]:
group_id = full_df['label_group'].unique()

x_train_idx, x_val_idx = train_test_split(group_id, test_size=0.2, random_state=101)

train_df = full_df[full_df['label_group'].isin(x_train_idx)].reset_index(drop=True)

val_df = full_df[full_df['label_group'].isin(x_val_idx)].reset_index(drop=True)

In [13]:
print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

assert len(train_df) + len(val_df) == len(full_df)

assert set(train_df['label_group']).isdisjoint(set(val_df['label_group']))

Train samples: 27393
Validation samples: 6857


## 5. Dataset & DataLoader

Prepare product images for EfficientNet-B0 feature extraction.

- Load images using OpenCV.
- Apply resizing and ImageNet normalization.
- Convert images to PyTorch tensors.
- Create DataLoaders without shuffling to preserve the order of posting IDs.

In [14]:
class ShopeeImageDataset(Dataset):
    def __init__(self, root, transform, df):
        super().__init__()

        self.df = df
        self.root = root
        self.images_path = self.df['image'].values
        self.transform = transform

    def __len__(self):
        return len(self.images_path)

    def __getitem__(self, index):
        img_path = os.path.join(
            self.root, 'train_images', self.images_path[index]
        )
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            img = self.transform(image=img)['image']

        return img

In [15]:
val_transforms = A.Compose([
    A.Resize(384, 384, p=1.0),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=255.0,
        p=1.0
    ),
    ToTensorV2(p=1.0)
])

In [16]:
train_dataset = ShopeeImageDataset(root, val_transforms, train_df)
val_dataset = ShopeeImageDataset(root, val_transforms, val_df)

train_loader = DataLoader(dataset=train_dataset,batch_size=32,shuffle=False)

val_loader = DataLoader(dataset=val_dataset,batch_size=32,shuffle=False)

## 6. Model Initialization

Initialize a pretrained EfficientNet-B0 model using `timm` for visual feature extraction.

- Pretrained ImageNet weights.
- Output embedding dimension: 1280.
- Model transferred to the selected device.

In [17]:
model = timm.create_model("efficientnet_b0", pretrained=True,num_classes=0)
model = model.to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

## 7. Feature Extraction

Extract 1280-dimensional visual embeddings from training and validation images using the pretrained EfficientNet-B0 model.
- Return NumPy embedding matrices for both datasets.

In [18]:
train_embeddings = get_embeddings(model, train_loader, device)
val_embeddings = get_embeddings(model, val_loader, device)

getting embeddings: 100%|██████████| 215/215 [01:09<00:00,  3.10it/s]


In [ ]:
print(f"Train embeddings: {train_embeddings.shape}")
print(f"Validation embeddings: {val_embeddings.shape}")

assert train_embeddings.shape == (len(train_df), 1280)
assert val_embeddings.shape == (len(val_df), 1280)

## 8. Saving & Verification

Save the extracted EfficientNet-B0 embeddings to Google Drive.
Each `.npz` file contains:
- `embeddings` — feature matrix (N, 1280).
- `posting_ids` — corresponding product identifiers.

Reload the saved files and verify their shapes and alignment with the original dataset.

In [ ]:
path_to_emb = '/content/drive/MyDrive/shopee/artifacts/embeddings/'
os.makedirs(path_to_emb, exist_ok=True)

In [ ]:
np.savez(os.path.join(path_to_emb, 'efficientnet_b0_train.npz'),
    embeddings=train_embeddings,
    posting_ids=train_df['posting_id'].values
)

np.savez(os.path.join(path_to_emb, 'efficientnet_b0_val.npz'),
    embeddings=val_embeddings,
    posting_ids=val_df['posting_id'].values
)

In [ ]:
train_embs = np.load(os.path.join(path_to_emb, 'efficientnet_b0_train.npz'))

val_embs = np.load(os.path.join(path_to_emb, 'efficientnet_b0_val.npz'))

print("Train:", train_embs['embeddings'].shape)
print("Validation:", val_embs['embeddings'].shape)

assert np.array_equal(train_embs['posting_ids'],train_df['posting_id'].values)

assert np.array_equal( val_embs['posting_ids'],val_df['posting_id'].values)